# 07 - Temporal Encoding Experiments

Explore different temporal encoding strategies for weather sequences.

## Encoding Strategies
1. Sinusoidal positional encoding (baseline)
2. Learned positional embeddings
3. Day-of-year + year encoding
4. Seasonal phase encoding
5. Relative position encoding

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import math
from pathlib import Path

RESULTS_DIR = Path('../../results/model_experiments')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Sinusoidal Encoding (Standard)

In [ ]:
class SinusoidalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    
    def __init__(self, d_model: int, max_len: int = 365):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## 2. Day-of-Year + Year Encoding

In [ ]:
class SeasonalEncoding(nn.Module):
    """
    Encoding that captures seasonal cycles explicitly.
    
    Uses day-of-year for seasonal position and optional year for trends.
    """
    
    def __init__(self, d_model: int, max_year_offset: int = 5):
        super().__init__()
        self.d_model = d_model
        
        # Day of year encoding (cyclic)
        self.doy_proj = nn.Linear(4, d_model // 2)  # sin/cos for day and week
        
        # Year offset encoding (for multi-year sequences)
        self.year_embed = nn.Embedding(max_year_offset * 2 + 1, d_model // 2)
    
    def forward(self, day_of_year: torch.Tensor, year_offset: torch.Tensor = None):
        """
        Args:
            day_of_year: [batch, seq] values 1-365
            year_offset: [batch, seq] year relative to reference (optional)
        """
        # Cyclic encoding of day of year
        doy_norm = day_of_year.float() / 365.0 * 2 * math.pi
        week_norm = day_of_year.float() / 7.0 * 2 * math.pi
        
        cyclic_features = torch.stack([
            torch.sin(doy_norm),
            torch.cos(doy_norm),
            torch.sin(week_norm),
            torch.cos(week_norm)
        ], dim=-1)
        
        doy_embed = self.doy_proj(cyclic_features)
        
        if year_offset is not None:
            year_embed = self.year_embed(year_offset + 5)  # Offset to positive
            return torch.cat([doy_embed, year_embed], dim=-1)
        else:
            # Pad if no year offset
            padding = torch.zeros(*doy_embed.shape[:-1], self.d_model // 2, 
                                 device=doy_embed.device)
            return torch.cat([doy_embed, padding], dim=-1)

# Test seasonal encoding
seasonal_enc = SeasonalEncoding(d_model=128)
doy = torch.arange(1, 366).unsqueeze(0)  # Full year
encoding = seasonal_enc(doy)

print(f"Seasonal encoding shape: {encoding.shape}")

## 3. Growing Season Phase Encoding

In [ ]:
class GrowingSeasonEncoding(nn.Module):
    """
    Encoding that emphasizes agricultural growing season phases.
    
    Phases:
    - Pre-planting (early spring)
    - Planting
    - Early growth
    - Peak growth
    - Senescence
    - Harvest
    - Dormant (winter)
    """
    
    def __init__(self, d_model: int, latitude_aware: bool = True):
        super().__init__()
        self.latitude_aware = latitude_aware
        
        # Phase embeddings (7 phases)
        self.phase_embed = nn.Embedding(7, d_model // 2)
        
        # Continuous encoding
        self.continuous_proj = nn.Linear(2, d_model // 2)
    
    def get_phase(self, day_of_year: torch.Tensor, latitude: torch.Tensor = None):
        """Determine growing season phase from day of year."""
        # Northern hemisphere defaults
        # Adjust boundaries based on latitude if available
        
        phase = torch.zeros_like(day_of_year)
        
        # Simplified phase assignment (Northern Hemisphere)
        phase = torch.where(day_of_year < 60, torch.tensor(6), phase)   # Dormant
        phase = torch.where((day_of_year >= 60) & (day_of_year < 100), 
                           torch.tensor(0), phase)  # Pre-planting
        phase = torch.where((day_of_year >= 100) & (day_of_year < 140), 
                           torch.tensor(1), phase)  # Planting
        phase = torch.where((day_of_year >= 140) & (day_of_year < 180), 
                           torch.tensor(2), phase)  # Early growth
        phase = torch.where((day_of_year >= 180) & (day_of_year < 240), 
                           torch.tensor(3), phase)  # Peak growth
        phase = torch.where((day_of_year >= 240) & (day_of_year < 280), 
                           torch.tensor(4), phase)  # Senescence
        phase = torch.where((day_of_year >= 280) & (day_of_year < 320), 
                           torch.tensor(5), phase)  # Harvest
        phase = torch.where(day_of_year >= 320, torch.tensor(6), phase)  # Dormant
        
        return phase.long()
    
    def forward(self, day_of_year: torch.Tensor, latitude: torch.Tensor = None):
        """Encode with growing season awareness."""
        phase = self.get_phase(day_of_year, latitude)
        phase_encoding = self.phase_embed(phase)
        
        # Continuous cyclic encoding
        doy_norm = day_of_year.float() / 365.0 * 2 * math.pi
        continuous = torch.stack([
            torch.sin(doy_norm),
            torch.cos(doy_norm)
        ], dim=-1)
        continuous_encoding = self.continuous_proj(continuous)
        
        return torch.cat([phase_encoding, continuous_encoding], dim=-1)

## 4. Visualization Comparison

In [ ]:
# Compare encodings visually
d_model = 64
days = torch.arange(1, 366)

# Generate encodings
sin_enc = SinusoidalEncoding(d_model)(torch.zeros(1, 365, d_model))
seasonal_enc = SeasonalEncoding(d_model)(days.unsqueeze(0))
growing_enc = GrowingSeasonEncoding(d_model)(days.unsqueeze(0))

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, enc, name in zip(axes, 
                         [sin_enc, seasonal_enc, growing_enc],
                         ['Sinusoidal', 'Seasonal', 'Growing Season']):
    im = ax.imshow(enc[0].detach().numpy().T, aspect='auto', cmap='RdBu')
    ax.set_ylabel('Dimension')
    ax.set_title(f'{name} Encoding')
    plt.colorbar(im, ax=ax)

axes[-1].set_xlabel('Day of Year')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'temporal_encoding_comparison.png', dpi=150)
plt.show()

## 5. Recommendations

For the Soil State Transformer, we recommend:

1. **Base encoding**: Sinusoidal for smooth interpolation
2. **Augmentation**: Add seasonal phase as discrete feature
3. **Multi-scale**: Combine day, week, month, and year cycles

Final encoder will combine multiple strategies.

In [ ]:
import json

recommendations = {
    'primary_encoding': 'Sinusoidal',
    'rationale': 'Smooth interpolation, proven in NLP transformers',
    'augmentations': [
        'Day-of-year cyclic features',
        'Growing season phase embedding',
        'Year offset for multi-year sequences'
    ],
    'implementation': 'Combine sinusoidal base with learned seasonal features'
}

with open(RESULTS_DIR / 'temporal_encoding_recommendations.json', 'w') as f:
    json.dump(recommendations, f, indent=2)

print("Recommendations saved")